# LC 295 — Find Median from Data Stream
**Day 52 | Heap Review | Difficulty: Hard**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Split the stream into two halves.
A max-heap (<code>lo</code>) holds the smaller half; a min-heap
(<code>hi</code>) holds the larger half. Rebalance after every
insert so the sizes differ by at most 1, and the median is always
at the tops.
</div>

## Official Problem Statement

The **median** is the middle value in an ordered integer list.
If the list size is even, the median is the mean of the two
middle values.

Implement the `MedianFinder` class:
- `MedianFinder()` initializes the object.
- `void addNum(int num)` adds integer `num` to the data structure.
- `double findMedian()` returns the median of current elements.

**Constraints:**
- `-10^5 <= num <= 10^5`
- At most `5 * 10^4` calls to `addNum` and `findMedian`.
- `findMedian` is called at least once after each `addNum`.

## What This Is Actually Asking

Numbers arrive one at a time with no knowledge of future values.
After each insertion we must report the current median in O(1).
Sorting after every insert is O(n log n) per call — too slow.
The key insight is that median only needs the boundary between
the lower half and the upper half, which two heaps track cheaply.
Python's `heapq` is a min-heap, so negate values for the max-heap.

## Walk Through an Example by Hand

```
Add 1:
  push 1 to lo → lo=[-1]  (max-heap: top=1)
  move lo-top to hi → lo=[], hi=[1]
  len(lo)<len(hi): move hi-top to lo → lo=[-1], hi=[]
  median = -lo[0] = 1

Add 2:
  push 2 to lo → lo=[-2,-1]  top=-1 means max=1? No:
  Actually push 2 → lo=[-2,-1], top of lo=-(-2)=wait...

Cleaner trace with add(1), add(2), add(3):
  After add(1): lo=[-1], hi=[]      median=1.0
  After add(2): lo=[-1], hi=[2]     median=(1+2)/2=1.5
  After add(3): lo=[-2,-1], hi=[3]  median=2.0
                 ^top=2
```

## The Picture

```
    lo (max-heap, negated)     hi (min-heap)
    ──────────────────────     ─────────────
    smaller half               larger half
    top = largest of lo        top = smallest of hi

    [ ..., 3, 4, 5 ]  |  [ 6, 7, 8, ... ]
                   ^  |  ^
                   └──┴── median lives here

    Even total → median = (-lo[0] + hi[0]) / 2
    Odd total  → median = -lo[0]   (lo has one extra)

    Invariant: len(lo) == len(hi)  OR  len(lo) == len(hi)+1
```

## When To Use This Pattern

- When you need the **median of a stream** (no random access),
  think **two heaps** (max-heap lo + min-heap hi).
- When a problem needs the **boundary between two sorted halves**
  to update in O(log n), think two heaps.
- When `findMedian` must be O(1) but `addNum` can be O(log n),
  this is the canonical solution.
- When the problem involves a **sliding window median**, extend
  this pattern with lazy deletion.

## The Approach

Always push a new number into `lo` (the max-heap, values negated).
Then immediately move the max of `lo` to `hi` so every element in
`lo` is <= every element in `hi`.
If `hi` becomes longer than `lo`, move the min of `hi` back to
`lo` to restore the size invariant (lo has equal or one more).
Median is then either -lo[0] alone (odd count) or the average
of -lo[0] and hi[0] (even count).

In [ ]:
import heapq
from typing import List

In [ ]:
def test_harness(MedianFinderClass):
    """
    Replays op sequences and checks findMedian output.
    Each case: (ops, args, expected_outputs)
    None in expected_outputs means no check (addNum calls).
    """
    cases = [
        (
            ["addNum","findMedian","addNum","findMedian",
             "addNum","findMedian"],
            [1, None, 2, None, 3, None],
            [None, 1.0, None, 1.5, None, 2.0],
            "basic 1,2,3"
        ),
        (
            ["addNum","addNum","findMedian",
             "addNum","findMedian"],
            [1, 2, None, 3, None],
            [None, None, 1.5, None, 2.0],
            "LeetCode example"
        ),
        (
            ["addNum","findMedian"],
            [42, None],
            [None, 42.0],
            "single element"
        ),
    ]
    passed = 0
    for ops, args, expected, label in cases:
        obj = MedianFinderClass()
        ok = True
        for op, arg, exp in zip(ops, args, expected):
            if op == "addNum":
                obj.addNum(arg)
            else:
                got = obj.findMedian()
                if got != exp:
                    print(
                        f"FAILED [{label}] findMedian: "
                        f"expected {exp}, got {got}"
                    )
                    ok = False
        if ok:
            passed += 1
            print(f"PASSED | {label}")
    print(f"\n{passed}/{len(cases)} tests passed.")

In [ ]:
class MedianFinder:
    """
    Find median from a data stream using two heaps.

    lo: max-heap (store negated values) — smaller half
    hi: min-heap — larger half

    Invariant after every addNum:
      len(lo) == len(hi) or len(lo) == len(hi) + 1
      max(lo) <= min(hi)

    addNum  : O(log n)
    findMedian: O(1)
    Space   : O(n)
    """

    def __init__(self):
        """Initialize two empty heaps."""
        print("  __init__: lo=[], hi=[]")
        pass  # TODO: self.lo = []; self.hi = []

    def addNum(self, num: int) -> None:
        """
        Add num to the structure, maintaining heap invariant.

        Steps:
          1. Push -num to lo (max-heap via negation).
          2. Move lo's max to hi (ensures lo <= hi).
          3. If len(hi) > len(lo): move hi's min back to lo.
        """
        print(f"  addNum({num})")
        pass  # TODO: implement

    def findMedian(self) -> float:
        """
        Return current median.

        If len(lo) == len(hi): return (-lo[0] + hi[0]) / 2
        Else: return float(-lo[0])
        """
        print(f"  findMedian -> lo={self.lo}, hi={self.hi}")
        pass  # TODO: implement

In [ ]:
# Uncomment and run when solution is ready
# test_harness(MedianFinder)

## Complexity

| Approach | addNum | findMedian | Space |
|---|---|---|---|
| Sort on each call | O(n log n) | O(1) | O(n) |
| Sorted list insert | O(n) | O(1) | O(n) |
| Two heaps (optimal) | O(log n) | O(1) | O(n) |

## Real World Connection

**Citi** monitors streaming trade latency and must report the
median response time at any moment for SLA dashboards.
The two-heap structure ingests each new latency measurement
in O(log n) and answers "current median?" in O(1) — perfect
for a live metrics endpoint polled every second.
On **AWS Kinesis** pipelines, a similar design appears in
stateful stream processors (e.g., Apache Flink) that maintain
percentile statistics over sliding event windows without
buffering all records in sorted order.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra